# Published results

> **Placeholder:** No canonical timings have been published. Smoke output is diagnostic only.

# Full-workflow estimation scaling

This authoritative notebook replaces the former standalone estimation baseline: `n_zones=1` is the canonical baseline. It scales exactly `[1, 10, 50, 100]` zones.

## Methodology and environment

Each zone is an independent prefixed deep copy of the complete translated 23-component/32-connection workflow. Every fit uses all 28 theta and all four measurements per zone with the source example's private/shared semantics and `n_warmup=20`; compatible roles use the batched model layout and functional execution. CPU rows use the eager backend and CUDA rows use the CUDA Graph backend.

The exact matrix is CPU: single-start SLSQP single shooting. CUDA includes single-start SLSQP, single-start custom batched SQP, and a separately labelled eight-start custom batched SQP throughput experiment. CPU custom SQP is excluded. CPU collocation is excluded. CUDA IPOPT collocation (`hessian="exact"`) is excluded from this suite: on the full-workflow model it does not reach shooting quality. Every CUDA arm uses `execution_mode="functional"` and `execution_backend="cuda_graph"`. In `mode="full"`, each configuration runs once until native convergence or a 300-iteration cap. Rows label convergence, parameter recovery, and post-fit prediction quality. Every prediction and observation is retained in a compressed artifact, with aggregate quantiles for temperature, CO2, damper, and valve signals. Raw rows are checkpointed after every case.

In [ ]:
# Colab bootstrap: configure a branch, tag, or immutable commit SHA.
import os
import pathlib
import subprocess
import sys


GIT_REF = "main"
REPOSITORY = "https://github.com/JBjoernskov/Twin4Build.git"
if "google.colab" in sys.modules:
    root = pathlib.Path("/content/Twin4Build")
    if not root.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(root)], check=True)
    subprocess.run(["git", "-C", str(root), "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", str(root), "checkout", GIT_REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(root)], check=True)
else:
    root = pathlib.Path.cwd()
    if root.name == "benchmarks":
        root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from benchmarks.common import (
    BenchmarkConfig,
    ESTIMATION_MATRIX,
    run_estimation_scaling,
    ZONE_COUNTS,
    environment_metadata,
    seed_everything,
    serialize_results,
)

config = BenchmarkConfig(mode=os.environ.get("T4B_BENCHMARK_MODE", "smoke"))
seed_everything(config.seed)
assert ZONE_COUNTS == [1, 10, 50, 100]
environment_metadata(GIT_REF)

## Batched functional solver semantics

SLSQP and custom SQP evaluate the complete batched workflow through sequential functional single shooting. CPU includes only single-start SLSQP with `execution_backend="eager"`. CUDA uses `execution_backend="cuda_graph"` for single-start SLSQP, single-start custom SQP, and a separately labelled eight-start custom SQP throughput experiment. CUDA Graph is an execution backend, not a mode or a Hessian option. Every method retains complete topology, all zone residuals, all 28 theta per zone, solver counters, parameter recovery, compressed post-fit prediction arrays, signal-quality quantiles, and explicit failure or skip reasons.

In [ ]:
print("Zone counts:", ZONE_COUNTS)
print("Matrix:", ESTIMATION_MATRIX)
rows = run_estimation_scaling(config)
rows

In [ ]:
result_path = serialize_results("estimation_scaling", config, rows, GIT_REF)
print(result_path)

## Interpretation

Plot time together with convergence, parameter recovery, and post-fit signal quality. Nonconvergence at the 300-iteration cap is a quality outcome, not a speedup. Compare CPU and CUDA only for matching single-start SLSQP rows; report the eight-start custom SQP arm separately from single-start timings. Component batching and untimed post-fit quality-rollout time remain separate. Prediction artifacts retain all observed and modeled temperature, CO2, damper, and valve values; table summaries use their scored-window quantiles and error metrics.